<a href="https://colab.research.google.com/github/shiqichen27/CIS3120/blob/main/Copy_of_CIS3120_Inheritance_Class_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inheritance and Object-Oriented Design

**Course:** CIS 3120 -- Programming for Analytics  
**Topic:** Runestone FOPP Chapter 22 -- Inheritance  
**Duration:** 75 minutes

---

## Learning Objectives

By the end of this class, you will be able to:

1. Define inheritance and explain why it reduces code duplication.
2. Create a subclass that inherits attributes and methods from a parent class.
3. Override methods in a subclass to customize behavior.
4. Use `super()` to invoke parent-class constructors and methods.
5. Distinguish between encapsulation (data hiding) and public interfaces.
6. Apply polymorphism by writing functions that operate on objects of different classes through a shared interface.
7. Compare composition and inheritance and select the appropriate design strategy for a given problem.

---

**Running theme:** All examples in this notebook use a simplified business analytics consultancy. We model employees, analysts, managers, and the reports they produce. This keeps the examples grounded in a domain you will encounter professionally.

---
## Part 1 -- Inheritance Basics (15 min)

### The Copy-Paste Problem

Suppose you are building a small HR analytics system. You need classes for different types of employees. Without inheritance, you might write something like this:

In [ ]:
# WITHOUT inheritance -- notice the duplication

class Analyst:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def annual_cost(self):
        return self.salary * 1.30  # salary + 30% benefits overhead

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"


class Manager:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def annual_cost(self):
        return self.salary * 1.30  # identical logic duplicated

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"  # identical logic duplicated


a = Analyst("Dana", "A101", 85000)
m = Manager("Lee", "M201", 110000)
print(a, "->", a.annual_cost())
print(m, "->", m.annual_cost())

In [ ]:
FinanceAnalyst1 = Analyst("Dana", "A101", 85000)

In [ ]:
FinanceAnalyst1.name

In [ ]:
print(FinanceAnalyst1)

In [ ]:
MarketingAnalyst7 = Analyst("Lee", "M201", 110000)

In [ ]:
MarketingAnalyst7.name

In [ ]:
print(MarketingAnalyst7)

The `Analyst` and `Manager` classes share identical code for `__init__`, `annual_cost`, and `__str__`. If we later change the benefits multiplier from 1.30 to 1.35, we must remember to update it in every class. This is error-prone and does not scale.

### Defining a Base Class and a Subclass

Inheritance lets us define common behavior once in a **base class** (also called a parent class or superclass) and let **subclasses** (child classes) reuse that behavior automatically.

In [ ]:
class Employee:
    """Base class for all employees in the consultancy."""

    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def annual_cost(self):
        """Total annual cost including a 30% benefits overhead."""
        return self.salary * 1.30

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"


class Analyst(Employee):
    """An Analyst is an Employee. No additional behavior yet."""
    pass

class Manager(Employee):
    """A Manager is an Employee. No additional behavior yet."""
    pass


# Create an Analyst and confirm it has inherited everything from Employee
a = Analyst("Dana", "A101", 85000)

print(a)                  # inherited __str__
print(a.annual_cost())    # inherited annual_cost
print(a.name)             # inherited attribute

In [ ]:
a.skill = ['Python', 'R', 'Excel']

In [ ]:
a.skill

In [ ]:
a.name

In [ ]:
a.name = 'Priya'

In [ ]:
a.name

In [ ]:
a.name.lower()

By writing `class Analyst(Employee)`, we declare that `Analyst` **is a** kind of `Employee`. The `Analyst` class automatically receives all methods and the `__init__` defined in `Employee` without any additional code.

### How Python Resolves Methods

When you call `a.annual_cost()`, Python looks for `annual_cost` in the following order:

1. The instance `a` itself.
2. The class `Analyst`.
3. The parent class `Employee`.
4. Continuing up the chain until `object` (the ultimate base class in Python).

This lookup sequence is called the **Method Resolution Order (MRO)**. You can inspect it:

In [ ]:
print(Analyst.__mro__)

### Exercise 1 -- Create a Subclass

**Your Turn:** Create a `Manager` class that inherits from `Employee`. Then create a `Manager` instance and verify that it has access to the inherited `annual_cost` method.

In [28]:
# EXERCISE 1
# TODO: Define a Manager class that inherits from Employee.
# TODO: Create an instance with name="Lee", employee_id="M201", salary=110000.
# TODO: Print the instance and its annual_cost().

# --- Your code below ---
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def annual_cost(self):
        return self.salary * 1.30

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"

class Manager(Employee):
    pass

m = Manager("Lee", "M201", 110000)
print(m)
print("Annual cost: ", m.annual_cost())


Lee (ID: M201)
Annual cost:  143000.0


<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
class Manager(Employee):
    pass

m = Manager("Lee", "M201", 110000)
print(m)
print(m.annual_cost())
```

Expected output:
```
Lee (ID: M201)
143000.0
```
</details>

---
## Part 2 -- Method Overriding (10 min)

Inheritance is useful, but subclasses often need to **customize** inherited behavior. A `Manager` might have a different string representation that includes their title, or a `SeniorAnalyst` might have a higher benefits multiplier. We accomplish this through **method overriding**: defining a method in the subclass with the same name as the one in the parent class.

In [26]:
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def annual_cost(self):
        return self.salary * 1.30

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"


class SeniorAnalyst(Employee):
    """Senior analysts have a higher benefits multiplier (35%)."""

    def annual_cost(self):
        # This OVERRIDES the parent version
        return self.salary * 1.35


class Manager(Employee):
    """Managers display their title in their string representation."""

    def __init__(self, name, employee_id, salary, direct_reports=0):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary
        self.direct_reports = direct_reports

    def __str__(self):
        # This OVERRIDES the parent version
        return f"Manager {self.name} (ID: {self.employee_id}, reports: {self.direct_reports})"


sa = SeniorAnalyst("Priya", "A301", 105000)
print(sa, "-> annual cost:", sa.annual_cost())  # uses overridden annual_cost

m = Manager("Lee", "M201", 110000, direct_reports=5)
print(m)  # uses overridden __str__
print("Annual cost:", m.annual_cost())  # uses INHERITED annual_cost (not overridden)

Priya (ID: A301) -> annual cost: 141750.0
Manager Lee (ID: M201, reports: 5)
Annual cost: 143000.0


Notice that `Manager` overrides `__str__` but does **not** override `annual_cost`, so it still uses the parent's version. Each method is resolved independently.

### Exercise 2 -- Override a Method

**Your Turn:** Modify the `Manager` class below so that its `__str__` method returns the format: `"Manager <name> | <direct_reports> direct reports"`. Do not change the `Employee` base class.

In [15]:
# EXERCISE 2

class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def __str__(self):
        return f"Manager {self.name} | {self.direct_reports} direct reports"


class Manager(Employee):
    def __init__(self, name, employee_id, salary, direct_reports=0):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary
        self.direct_reports = direct_reports

    # TODO: Override __str__ so it returns:
    #   "Manager <name> | <direct_reports> direct reports"

# --- Verification ---
m = Manager("Lee", "M201", 110000, direct_reports=5)
print(m)  # Expected: Manager Lee | 5 direct reports

Manager Lee | 5 direct reports


<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
class Manager(Employee):
    def __init__(self, name, employee_id, salary, direct_reports=0):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary
        self.direct_reports = direct_reports

    def __str__(self):
        return f"Manager {self.name} | {self.direct_reports} direct reports"

m = Manager("Lee", "M201", 110000, direct_reports=5)
print(m)  # Manager Lee | 5 direct reports
```
</details>

---
## Part 3 -- The `super()` Function (10 min)

Look at the `Manager.__init__` in the previous example. It manually sets `self.name`, `self.employee_id`, and `self.salary` -- the exact same work that `Employee.__init__` already does. This is the copy-paste problem reappearing inside `__init__`.

The `super()` function solves this by giving you a reference to the parent class, so you can delegate work to it.

In [2]:
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def annual_cost(self):
        return self.salary * 1.30

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"


class Manager(Employee):
    def __init__(self, name, employee_id, salary, direct_reports=0):
        # Delegate the common initialization to the parent class
        super().__init__(name, employee_id, salary)
        # Then handle Manager-specific attributes
        self.direct_reports = direct_reports

    def __str__(self):
        return f"Manager {self.name} (ID: {self.employee_id}, reports: {self.direct_reports})"


m = Manager("Lee", "M201", 110000, direct_reports=5)
print(m)
print("Salary attr:", m.salary)  # set by Employee.__init__ via super()

Manager Lee (ID: M201, reports: 5)
Salary attr: 110000


### Using `super()` in Overridden Methods

`super()` is not limited to `__init__`. You can use it in any overridden method when you want to **extend** the parent behavior rather than completely replace it.

In [3]:
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def summary(self):
        return f"{self.name} (ID: {self.employee_id}), Salary: ${self.salary:,.0f}"


class Manager(Employee):
    def __init__(self, name, employee_id, salary, direct_reports=0):
        super().__init__(name, employee_id, salary)
        self.direct_reports = direct_reports

    def summary(self):
        # Start with everything the parent summary provides ...
        base = super().summary()
        # ... then append Manager-specific information
        return f"{base}, Direct Reports: {self.direct_reports}"


m = Manager("Lee", "M201", 110000, direct_reports=5)
print(m.summary())

Lee (ID: M201), Salary: $110,000, Direct Reports: 5


### Exercise 3 -- Refactor with `super()`

**Your Turn:** The `SeniorAnalyst` class below duplicates the parent's `__init__` logic. Refactor it to use `super().__init__()` instead.

In [4]:
# EXERCISE 3

class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary


class SeniorAnalyst(Employee):
    def __init__(self, name, employee_id, salary, specialty):
        # TODO: Replace the three lines below with a single super().__init__() call.
        super().__init__(name, employee_id, salary)
        self.specialty = specialty


# --- Verification ---
sa = SeniorAnalyst("Priya", "A301", 105000, "NLP")
print(sa.name, sa.employee_id, sa.salary, sa.specialty)

Priya A301 105000 NLP


<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
class SeniorAnalyst(Employee):
    def __init__(self, name, employee_id, salary, specialty):
        super().__init__(name, employee_id, salary)
        self.specialty = specialty

sa = SeniorAnalyst("Priya", "A301", 105000, "NLP")
print(sa.name, sa.employee_id, sa.salary, sa.specialty)
# Priya A301 105000 NLP
```
</details>

---
## Part 4 -- Encapsulation (10 min)

**Encapsulation** is the practice of restricting direct access to an object's internal data and instead providing controlled access through methods. The goal is to protect the integrity of the object's state.

### The Problem: Unrestricted Access

Consider what happens when there are no restrictions on attribute access:

In [ ]:
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

e = Employee("Dana", "A101", 85000)

# Nothing prevents an invalid assignment
e.salary = -50000
print(f"{e.name} salary: ${e.salary:,.0f}")  # negative salary -- clearly invalid

### Python's Convention: The Underscore Prefix

Python does not enforce strict access control the way some other languages do. Instead, it relies on a naming convention:

- `self.salary` -- a **public** attribute; external code is expected to access it freely.
- `self._salary` -- a **protected** attribute by convention; a signal to other developers that this attribute is internal and should not be accessed directly.
- `self.__salary` -- triggers **name mangling**; Python renames it to `_ClassName__salary` to make accidental access harder (but not impossible).

### Using Properties for Controlled Access

The `@property` decorator lets you define getter and setter methods that behave like ordinary attribute access from the caller's perspective, while giving you a place to add validation logic.

In [ ]:
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self._salary = salary  # store in a "protected" attribute

    @property
    def salary(self):
        """Read the salary value."""
        return self._salary

    @salary.setter
    def salary(self, value):
        """Set the salary, but reject non-positive values."""
        if value <= 0:
            raise ValueError(f"Salary must be positive. Received: {value}")
        self._salary = value

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id}), Salary: ${self._salary:,.0f}"


e = Employee("Dana", "A101", 85000)
print(e)

# Valid update
e.salary = 90000
print("Updated:", e)

# Invalid update -- this will raise a ValueError
try:
    e.salary = -50000
except ValueError as err:
    print(f"Blocked: {err}")

From the outside, `e.salary = 90000` looks like a normal attribute assignment. But behind the scenes, the `@salary.setter` method runs and validates the input. This is encapsulation in action: the internal representation (`_salary`) is protected, and external access goes through a controlled interface.

### Exercise 4 -- Add a Validated Property

**Your Turn:** Add a `budget` property to the `Project` class below. The budget must be non-negative. If someone attempts to set a negative budget, raise a `ValueError`.

In [34]:
# EXERCISE 4

class Project:
    def __init__(self, title, budget):
        self.title = title
        self._budget = 0
        # TODO: Store budget in a protected attribute (self._budget)
        #       and add @property and @budget.setter with validation.
        self.budget = budget  # replace this line as part of your solution

    # TODO: Define the @property for budget
    @property
    def budget(self):
      return self._budget

    # TODO: Define the @budget.setter with a non-negative check
    @budget.setter
    def budget(self, value):
        if value < 0:
            raise ValueError
        self._salary = value


# --- Verification ---
p = Project("Q4 Dashboard", 25000)
print(f"{p.title}: ${p.budget:,.0f}")

p.budget = 30000
print(f"Updated: ${p.budget:,.0f}")

try:
    p.budget = -100
except ValueError as err:
    print(f"Blocked: {err}")

Q4 Dashboard: $0
Updated: $0
Blocked: 


<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
class Project:
    def __init__(self, title, budget):
        self.title = title
        self._budget = 0       # initialize before using the setter
        self.budget = budget    # use the setter for validation on construction

    @property
    def budget(self):
        return self._budget

    @budget.setter
    def budget(self, value):
        if value < 0:
            raise ValueError(f"Budget must be non-negative. Received: {value}")
        self._budget = value

p = Project("Q4 Dashboard", 25000)
print(f"{p.title}: ${p.budget:,.0f}")       # Q4 Dashboard: $25,000
p.budget = 30000
print(f"Updated: ${p.budget:,.0f}")          # Updated: $30,000

try:
    p.budget = -100
except ValueError as err:
    print(f"Blocked: {err}")                  # Blocked: Budget must be non-negative. Received: -100
```
</details>

---
## Part 5 -- Polymorphism (15 min)

**Polymorphism** means "many forms." In object-oriented programming, it refers to the ability to use the same interface (method name) on objects of different types, with each type providing its own implementation.

### Example: Report Generators

Imagine the consultancy produces several types of reports. Each report class has a `.generate()` method, but the output differs by type.

In [35]:
class SalesReport:
    def __init__(self, region, revenue):
        self.region = region
        self.revenue = revenue

    def generate(self):
        return f"[Sales Report] {self.region}: ${self.revenue:,.0f} in revenue"


class RiskReport:
    def __init__(self, category, score):
        self.category = category
        self.score = score

    def generate(self):
        return f"[Risk Report] {self.category}: risk score = {self.score}/10"


class InventoryReport:
    def __init__(self, warehouse, item_count):
        self.warehouse = warehouse
        self.item_count = item_count

    def generate(self):
        return f"[Inventory Report] {self.warehouse}: {self.item_count:,} items in stock"


# Create a list containing different report types
reports = [
    SalesReport("Northeast", 1_250_000),
    RiskReport("Cybersecurity", 8),
    InventoryReport("Warehouse A", 45_200),
    SalesReport("West Coast", 980_000),
]

# Polymorphism: the same .generate() call works on every object,
# but each object produces output appropriate to its type.
for report in reports:
    print(report.generate())

[Sales Report] Northeast: $1,250,000 in revenue
[Risk Report] Cybersecurity: risk score = 8/10
[Inventory Report] Warehouse A: 45,200 items in stock
[Sales Report] West Coast: $980,000 in revenue


### Why This Matters

The `for` loop above does not need to know the specific type of each report. It only needs to know that every object in the list has a `.generate()` method. This is the core value of polymorphism: you can write **general-purpose code** that works with any object conforming to a shared interface.

### Duck Typing

Python implements polymorphism through **duck typing**: "If it walks like a duck and quacks like a duck, then it is a duck." There is no requirement that the report classes share a common base class. They simply need to implement the same method name.

In [36]:
def print_all_reports(report_list):
    """Print any collection of objects that have a .generate() method."""
    print("=" * 60)
    print("CONSOLIDATED REPORT")
    print("=" * 60)
    for r in report_list:
        print(r.generate())
    print("=" * 60)


# This function works with ANY objects that have .generate(),
# regardless of their class.
print_all_reports(reports)

CONSOLIDATED REPORT
[Sales Report] Northeast: $1,250,000 in revenue
[Risk Report] Cybersecurity: risk score = 8/10
[Inventory Report] Warehouse A: 45,200 items in stock
[Sales Report] West Coast: $980,000 in revenue


### Polymorphism with Inheritance

Polymorphism is especially clean when the classes share a base class. This makes the contract explicit: every subclass is expected to provide its own version of the method.

In [37]:
class Report:
    """Base class that defines the interface for all reports."""

    def generate(self):
        raise NotImplementedError("Subclasses must implement generate()")


class SalesReport(Report):
    def __init__(self, region, revenue):
        self.region = region
        self.revenue = revenue

    def generate(self):
        return f"[Sales] {self.region}: ${self.revenue:,.0f}"


class RiskReport(Report):
    def __init__(self, category, score):
        self.category = category
        self.score = score

    def generate(self):
        return f"[Risk] {self.category}: score {self.score}/10"


# If someone forgets to implement generate(), they get a clear error:
class IncompleteReport(Report):
    pass

try:
    bad = IncompleteReport()
    bad.generate()
except NotImplementedError as err:
    print(f"Error caught: {err}")

Error caught: Subclasses must implement generate()


### Exercise 5 -- Apply Polymorphism

**Your Turn:** Complete the `ComplianceReport` class and the `generate_summary` function below.

1. `ComplianceReport` should have a `.summary()` method that returns `"Compliance: <regulation> -- <status>"`.
2. `generate_summary` should accept a list of objects, call `.summary()` on each, and return a list of strings.

In [ ]:
# EXERCISE 5

class FinancialReport:
    def __init__(self, quarter, profit):
        self.quarter = quarter
        self.profit = profit

    def summary(self):
        return f"Finance: {self.quarter} -- Profit ${self.profit:,.0f}"


class ComplianceReport:
    def __init__(self, regulation, status):
        self.regulation = regulation
        self.status = status

    # TODO: Define a summary() method that returns:
    #   "Compliance: <regulation> -- <status>"


def generate_summary(report_list):
    # TODO: Return a list of strings by calling .summary() on each item.
    pass


# --- Verification ---
items = [
    FinancialReport("Q3 2025", 320000),
    ComplianceReport("SOX", "Compliant"),
    ComplianceReport("GDPR", "Under Review"),
]

results = generate_summary(items)
for line in results:
    print(line)

# Expected output:
# Finance: Q3 2025 -- Profit $320,000
# Compliance: SOX -- Compliant
# Compliance: GDPR -- Under Review

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
class ComplianceReport:
    def __init__(self, regulation, status):
        self.regulation = regulation
        self.status = status

    def summary(self):
        return f"Compliance: {self.regulation} -- {self.status}"


def generate_summary(report_list):
    return [item.summary() for item in report_list]

items = [
    FinancialReport("Q3 2025", 320000),
    ComplianceReport("SOX", "Compliant"),
    ComplianceReport("GDPR", "Under Review"),
]

results = generate_summary(items)
for line in results:
    print(line)
```
</details>

---
## Part 6 -- Composition vs. Inheritance (10 min)

### Two Ways to Reuse Code

So far we have focused on **inheritance** ("is-a" relationships). But there is another powerful technique: **composition** ("has-a" relationships).

| Inheritance ("is-a") | Composition ("has-a") |
|---|---|
| A `Manager` **is an** `Employee` | A `Team` **has** `Employee` objects |
| Subclass inherits parent behavior | Object contains other objects as attributes |
| Creates tight coupling between classes | Creates loose coupling between classes |

### When Inheritance Causes Problems

Consider this scenario: you want to model a consulting team. Should `Team` inherit from `Employee`?

In [ ]:
# PROBLEMATIC: using inheritance where composition is more appropriate

class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary


class Team(Employee):  # a Team "is an" Employee? That does not make sense.
    def __init__(self, team_name, members):
        # What would we pass for employee_id or salary?
        super().__init__(team_name, "N/A", 0)
        self.members = members


# This works, but it is misleading:
t = Team("Data Science Pod", ["Dana", "Priya", "Lee"])
print(t.employee_id)  # "N/A" -- meaningless for a team
print(t.salary)       # 0 -- a team does not have a personal salary

The `Team` class inherits attributes (`employee_id`, `salary`) that have no meaning for a team. This is a sign that inheritance is the wrong relationship.

### Composition: The Better Fit

In [ ]:
class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"


class Team:
    """A Team HAS employees. It does not inherit from Employee."""

    def __init__(self, team_name):
        self.team_name = team_name
        self.members = []  # composition: a list of Employee objects

    def add_member(self, employee):
        self.members.append(employee)

    def total_salary_cost(self):
        return sum(emp.salary for emp in self.members)

    def roster(self):
        return [str(emp) for emp in self.members]

    def __str__(self):
        return f"Team '{self.team_name}' ({len(self.members)} members)"


# Build a team using composition
team = Team("Analytics Pod")
team.add_member(Employee("Dana", "A101", 85000))
team.add_member(Employee("Priya", "A301", 105000))
team.add_member(Employee("Lee", "M201", 110000))

print(team)
print("Roster:", team.roster())
print(f"Total salary cost: ${team.total_salary_cost():,.0f}")

### Guidelines for Choosing

Use **inheritance** when the subclass genuinely **is a** specialized version of the parent. The subclass should be substitutable anywhere the parent is expected.

Use **composition** when one object **contains** or **uses** other objects. The contained objects have their own independent identity and lifecycle.

A useful heuristic: if you find yourself passing meaningless values to the parent's `__init__` (as in the `Team(Employee)` example above), composition is likely the better design.

### Exercise 6 -- Refactor to Composition

**Your Turn:** The code below uses inheritance to model a `Department` as a kind of `Employee`. Refactor it so that `Department` uses composition instead. A `Department` should contain a list of `Employee` objects and provide a `headcount()` method that returns the number of employees.

In [ ]:
# EXERCISE 6

class Employee:
    def __init__(self, name, employee_id, salary):
        self.name = name
        self.employee_id = employee_id
        self.salary = salary

    def __str__(self):
        return f"{self.name} (ID: {self.employee_id})"


# CURRENT (problematic) design -- Department inherits from Employee
class Department(Employee):
    def __init__(self, dept_name, employees):
        super().__init__(dept_name, "N/A", 0)
        self.employees = employees

    def headcount(self):
        return len(self.employees)


# TODO: Rewrite the Department class so that it does NOT inherit from Employee.
#       It should:
#         - Accept dept_name in __init__
#         - Have an add_employee(emp) method
#         - Have a headcount() method returning the number of employees
#         - Have a __str__ that returns "Department: <dept_name> (<headcount> employees)"

# --- Your code below ---


# --- Verification (uncomment after writing your solution) ---
# dept = Department("Business Intelligence")
# dept.add_employee(Employee("Dana", "A101", 85000))
# dept.add_employee(Employee("Priya", "A301", 105000))
# print(dept)            # Expected: Department: Business Intelligence (2 employees)
# print(dept.headcount())  # Expected: 2

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
class Department:
    def __init__(self, dept_name):
        self.dept_name = dept_name
        self.employees = []

    def add_employee(self, emp):
        self.employees.append(emp)

    def headcount(self):
        return len(self.employees)

    def __str__(self):
        return f"Department: {self.dept_name} ({self.headcount()} employees)"


dept = Department("Business Intelligence")
dept.add_employee(Employee("Dana", "A101", 85000))
dept.add_employee(Employee("Priya", "A301", 105000))
print(dept)              # Department: Business Intelligence (2 employees)
print(dept.headcount())  # 2
```
</details>

---
## Wrap-Up (5 min)

### Key Takeaways

**Inheritance** allows a subclass to reuse the attributes and methods of a parent class, eliminating code duplication. Use it when the subclass genuinely "is a" specialized version of the parent.

**Method overriding** lets a subclass replace or customize an inherited method. Python resolves methods by searching the subclass first, then the parent, following the Method Resolution Order.

**`super()`** provides a reference to the parent class, letting you delegate work (especially `__init__` logic) rather than duplicating it. You can also use `super()` in overridden methods to extend parent behavior.

**Encapsulation** protects an object's internal state by routing access through properties or methods that enforce validation rules. Python uses naming conventions (single and double underscores) rather than strict access modifiers.

**Polymorphism** enables you to write general-purpose code that operates on objects of different types through a shared method interface. Python's duck typing means that any object with the right method can participate, regardless of its class hierarchy.

**Composition** models "has-a" relationships by storing objects as attributes. When inheritance produces a forced or misleading class hierarchy, composition is usually the better design.

### Review Questions

1. What is the difference between overriding a method and overloading a method?
2. Why does `super().__init__()` reduce the risk of bugs when a parent class changes?
3. Give an example of a real-world "has-a" relationship that should use composition, not inheritance.
4. Can polymorphism work without inheritance in Python? Why or why not?

### Further Reading

- Runestone Interactive: *Foundations of Python Programming*, Chapter 22
- Python documentation: [Classes](https://docs.python.org/3/tutorial/classes.html)
- Real Python: [Inheritance and Composition](https://realpython.com/inheritance-composition-python/)